In [ ]:
# =============================================================================
# ULTRASOUND MUSCLE SEGMENTATION — CLASSICAL ML PIPELINE v5
# Kaggle Notebook | Rectus Femoris ACSA Computation
# Reference: Noble & Boukerroui (2006) IEEE TMI


# SECTION 1: IMPORTS


import numpy as np
import pandas as pd
import cv2
import warnings
import json
import time
import random
from pathlib import Path
from IPython.display import display as ipy_display

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from skimage.morphology import disk, binary_closing, remove_small_objects
from skimage.measure import label, regionprops
from skimage.filters.rank import entropy as rank_entropy
from skimage.feature import graycomatrix, graycoprops

from scipy.ndimage import binary_fill_holes, distance_transform_edt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

import joblib

warnings.filterwarnings('ignore')


# SECTION 2: CONFIGURATION


class Config:

    BASE_DIR = Path("/kaggle/input/datasets/aashakaashara/rf-input/rf_inputs")
    IMAGE_DIR = BASE_DIR / "images"
    MASK_DIR = BASE_DIR / "segmentation_masks"
    METADATA_PATH = BASE_DIR / "Results.xlsx"
    FAILED_PATH = BASE_DIR / "failed_images.txt"
    OUTPUT_DIR = Path("/kaggle/working/outputs")
    MODEL_PATH = OUTPUT_DIR / "rf_model_v5.joblib"
    RESULTS_PATH = OUTPUT_DIR / "results_v5.json"

    RANDOM_SEED   = 42

    TRAIN_RATIO = 0.70
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15

    SCAN_THRESHOLD = 10
    MEDIAN_KSIZE = 5
    CLAHE_CLIP = 2.0
    CLAHE_GRID = (8, 8)

    WINDOW_SMALL = 7
    WINDOW_MED = 21
    WINDOW_LARGE = 51
    WINDOW_XL = 71
    TEXTURE_RADIUS = 15      # rank entropy disk radius
    GRAD_SIGMA_FINE = 1       # Gaussian sigma before fine-scale Sobel
    GRAD_SIGMA_COARSE = 3     # Gaussian sigma before coarse-scale Sobel
    GLCM_PATCH_SIZE = 32      # non-overlapping patch size for GLCM
    GLCM_DISTANCES = [1]     # GLCM distances
    GLCM_ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4]  # 4 directions

    N_MUSCLE = 1500
    N_BOUNDARY_NEG = 1050    # 5px <= dist <= 50px from mask
    N_RANDOM_NEG = 450     # dist > 50px from mask
    BOUNDARY_MIN = 5
    BOUNDARY_MAX = 50

    RF_N_ESTIMATORS = 200
    RF_MAX_DEPTH = 20
    RF_MIN_SAMPLES_LEAF = 4
    RF_N_JOBS = -1

    MORPH_CLOSE_RADIUS = 5
    MIN_COMPONENT_SIZE = 500


cfg = Config()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(cfg.RANDOM_SEED)
random.seed(cfg.RANDOM_SEED)

print("✓ Configuration loaded")
print(f"  Sampling : {cfg.N_MUSCLE} muscle | "
      f"{cfg.N_BOUNDARY_NEG} boundary neg | "
      f"{cfg.N_RANDOM_NEG} random neg = "
      f"{cfg.N_MUSCLE+cfg.N_BOUNDARY_NEG+cfg.N_RANDOM_NEG} per image")



# SECTION 3: DATA LOADING & SPLIT

def load_failed_images(path: Path) -> set:
    failed = set()
    if path.exists():
        with open(path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                # Format: "Scalingline not found in C:/ML_PRO\rectus_img_352.tif"
                last_token = line.split()[-1]
                last_token = last_token.replace('\\', '/')
                filename   = last_token.split('/')[-1]
                stem       = Path(filename).stem
                failed.add(filename)
                failed.add(stem)
        print(f"  Excluded : {len(failed)//2} failed images")
    else:
        print(f"  WARNING  : failed_images.txt not found at {path}")
    return failed


def load_metadata(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Metadata not found: {path}")
    df = pd.read_excel(path)
    df.columns = [c.strip() for c in df.columns]
    df['stem'] = df['File'].astype(str).str.strip()
    print(f"  Metadata : {len(df)} rows")
    return df


def get_image_pairs(image_dir: Path, mask_dir: Path,
                    failed_set: set) -> list:
    pairs      = []
    valid_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    for img_path in sorted(image_dir.iterdir()):
        if img_path.suffix.lower() not in valid_exts:
            continue
        if img_path.name in failed_set or img_path.stem in failed_set:
            continue
        mask_path = None
        for ext in valid_exts:
            c = mask_dir / (img_path.stem + ext)
            if c.exists():
                mask_path = c
                break
        if mask_path is not None:
            pairs.append((img_path, mask_path))
    print(f"  Valid pairs: {len(pairs)}")
    return pairs


def split_pairs(pairs, train_r, val_r, seed):
    test_r     = 1.0 - train_r - val_r
    train, tmp = train_test_split(pairs, test_size=(val_r + test_r),
                                  random_state=seed)
    val, test  = train_test_split(tmp,
                                  test_size=(test_r / (val_r + test_r)),
                                  random_state=seed)
    return train, val, test


def load_image_raw(path: Path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError(f"Cannot read: {path}")
    return img


def load_mask_raw(path: Path) -> np.ndarray:
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise IOError(f"Cannot read: {path}")
    return (mask > 127).astype(np.uint8)


# Run
print("\n[DATA LOADING]")
failed_set  = load_failed_images(cfg.FAILED_PATH)
metadata    = load_metadata(cfg.METADATA_PATH)
all_pairs   = get_image_pairs(cfg.IMAGE_DIR, cfg.MASK_DIR, failed_set)
train_pairs, val_pairs, test_pairs = split_pairs(
    all_pairs, cfg.TRAIN_RATIO, cfg.VAL_RATIO, cfg.RANDOM_SEED
)
print(f"\n  Image-level split:")
print(f"Train : {len(train_pairs)}  ({len(train_pairs)/len(all_pairs)*100:.1f}%)")
print(f"Val : {len(val_pairs)}   ({len(val_pairs)/len(all_pairs)*100:.1f}%)")
print(f"Test : {len(test_pairs)}   ({len(test_pairs)/len(all_pairs)*100:.1f}%)  <- held-out")


# SECTION 4: PREPROCESSING


def preprocess(img_raw: np.ndarray,
               mask_raw: np.ndarray = None) -> dict:
    """
    1. Scan mask: pixels > SCAN_THRESHOLD
    2. Crop to bounding box
    3. Median filter (speckle)
    4. CLAHE (contrast)
    5. Normalise to float32 [0, 1]
    """
    orig_shape = img_raw.shape
    scan_full  = img_raw > cfg.SCAN_THRESHOLD

    rows = np.where(scan_full.any(axis=1))[0]
    cols = np.where(scan_full.any(axis=0))[0]
    if len(rows) == 0 or len(cols) == 0:
        r0, r1, c0, c1 = 0, orig_shape[0], 0, orig_shape[1]
    else:
        r0, r1 = int(rows[0]), int(rows[-1]) + 1
        c0, c1 = int(cols[0]), int(cols[-1]) + 1

    img_crop = img_raw[r0:r1, c0:c1]
    scan_crop = scan_full[r0:r1, c0:c1]

    img_med = cv2.medianBlur(img_crop, cfg.MEDIAN_KSIZE)
    clahe = cv2.createCLAHE(clipLimit=cfg.CLAHE_CLIP,
                                tileGridSize=cfg.CLAHE_GRID)
    img_cl = clahe.apply(img_med)
    img_proc = img_cl.astype(np.float32) / 255.0

    result = {
        'img_proc': img_proc,
        'scan_mask': scan_crop,
        'crop_bbox': (r0, r1, c0, c1),
        'orig_shape': orig_shape,
    }
    if mask_raw is not None:
        result['mask_cropped'] = mask_raw[r0:r1, c0:c1]
    return result


# SECTION 5: FEATURE EXTRACTION  (24 features per pixel)
#
# Group A — Local intensity stats (8)
#   mean_7, std_7, mean_21, std_21, mean_51, std_51, mean_71, std_71
#
# Group B — Intensity ratio (1)
#   mean_7 / (mean_71 + eps)
#
# Group C — Rank texture (2)
#   entropy_r15, variance_r15
#
# Group D — Multi-scale gradients + Laplacian (7)
#   gx_s1, gy_s1, gmag_s1 <- fine scale (=1)
#   gx_s3, gy_s3, gmag_s3 <- coarse scale (σ=3)
#   laplacian
#
# Group E — Patch GLCM texture (4)
#   glcm_contrast, glcm_homogeneity, glcm_energy, glcm_correlation
#
# Group F — Spatial (2)
#   dist_boundary, dist_center

FEATURE_NAMES = [
    # A
    'mean_7',    'std_7',
    'mean_21',   'std_21',
    'mean_51',   'std_51',
    'mean_71',   'std_71',
    # B
    'intensity_ratio',
    # C
    'entropy_r15',
    'variance_r15',
    # D
    'gx_s1', 'gy_s1', 'gmag_s1',
    'gx_s3', 'gy_s3', 'gmag_s3',
    'laplacian',
    # E
    'glcm_contrast',
    'glcm_homogeneity',
    'glcm_energy',
    'glcm_correlation',
    # F
    'dist_boundary',
    'dist_center',
]
N_FEATURES = len(FEATURE_NAMES)
assert N_FEATURES == 24, f"Expected 24 features, got {N_FEATURES}"


def _local_stats(img: np.ndarray, k: int):
    mean = cv2.boxFilter(img, -1, (k, k)).astype(np.float32)
    sq = cv2.boxFilter(img ** 2, -1, (k, k)).astype(np.float32)
    std = np.sqrt(np.clip(sq - mean ** 2, 0, None))
    return mean, std


def _compute_glcm_patch_features(img_8u: np.ndarray,
                                  patch_size: int) -> np.ndarray:
    """
    Divide image into non-overlapping patches of patch_size x patch_size.
    Compute GLCM for each patch (4 directions, averaged).
    Return 4 feature maps: contrast, homogeneity, energy, correlation.
    Each pixel gets the GLCM features of the patch it belongs to.

    Args:
        img_8u    : uint8 image (H, W)
        patch_size: size of each patch in pixels

    Returns:
        (H, W, 4) float32 array — 4 GLCM features per pixel
    """
    H, W = img_8u.shape
    n_feats = 4
    feat_maps = np.zeros((H, W, n_feats), dtype=np.float32)

    for r in range(0, H, patch_size):
        for c in range(0, W, patch_size):
            patch = img_8u[r:r+patch_size, c:c+patch_size]

            # Skip patches that are mostly black (outside scan cone)
            if patch.mean() < 5:
                continue

            # Reduce grey levels to speed up GLCM computation
            # 256 levels -> 32 levels
            patch_reduced = (patch // 8).astype(np.uint8)

            try:
                glcm = graycomatrix(
                    patch_reduced,
                    distances=cfg.GLCM_DISTANCES,
                    angles=cfg.GLCM_ANGLES,
                    levels=32,
                    symmetric=True,
                    normed=True,
                )
                contrast = float(graycoprops(glcm, 'contrast').mean())
                homogeneity = float(graycoprops(glcm, 'homogeneity').mean())
                energy = float(graycoprops(glcm, 'energy').mean())
                correlation = float(graycoprops(glcm, 'correlation').mean())
            except Exception:
                contrast = homogeneity = energy = correlation = 0.0

            # Assign to all pixels in this patch
            r_end = min(r + patch_size, H)
            c_end = min(c + patch_size, W)
            feat_maps[r:r_end, c:c_end, 0] = contrast
            feat_maps[r:r_end, c:c_end, 1] = homogeneity
            feat_maps[r:r_end, c:c_end, 2] = energy
            feat_maps[r:r_end, c:c_end, 3] = correlation

    # Normalise each GLCM feature map independently to [0, 1]
    for i in range(n_feats):
        fmap = feat_maps[:, :, i]
        fmin, fmax = fmap.min(), fmap.max()
        if fmax > fmin:
            feat_maps[:, :, i] = (fmap - fmin) / (fmax - fmin)

    return feat_maps


def extract_features(img_proc: np.ndarray,
                     scan_mask: np.ndarray) -> np.ndarray:
    """
    Build (H*W, 24) feature matrix from preprocessed image.
    """
    H, W   = img_proc.shape
    img_8u = (img_proc * 255).astype(np.uint8)

    #  Group A: local intensity stats
    mean_s,  std_s = _local_stats(img_proc, cfg.WINDOW_SMALL)
    mean_m,  std_m = _local_stats(img_proc, cfg.WINDOW_MED)
    mean_l,  std_l = _local_stats(img_proc, cfg.WINDOW_LARGE)
    mean_xl, std_xl = _local_stats(img_proc, cfg.WINDOW_XL)

    #  Group B: intensity ratio
    intensity_ratio = np.clip(mean_s / (mean_xl + 1e-8), 0, 5).astype(np.float32)

    #  Group C: rank texture
    selem = disk(cfg.TEXTURE_RADIUS)
    f_entropy = rank_entropy(img_8u, selem).astype(np.float32)
    f_entropy = f_entropy / (f_entropy.max() + 1e-8)
    _, f_var  = _local_stats(img_proc, cfg.TEXTURE_RADIUS * 2 + 1)

    #  Group D: multi-scale gradients + laplacian
    # Fine scale (σ=1)
    img_g1 = cv2.GaussianBlur(img_8u, (0, 0), sigmaX=cfg.GRAD_SIGMA_FINE)
    gx1 = cv2.Sobel(img_g1, cv2.CV_32F, 1, 0, ksize=3)
    gy1 = cv2.Sobel(img_g1, cv2.CV_32F, 0, 1, ksize=3)
    gmag1 = np.sqrt(gx1**2 + gy1**2).astype(np.float32)
    gx1_n = (gx1 / (np.abs(gx1).max() + 1e-8)).astype(np.float32)
    gy1_n = (gy1 / (np.abs(gy1).max() + 1e-8)).astype(np.float32)
    gmag1_n = (gmag1 / (gmag1.max() + 1e-8)).astype(np.float32)

    # Coarse scale (σ=3)
    img_g3 = cv2.GaussianBlur(img_8u, (0, 0), sigmaX=cfg.GRAD_SIGMA_COARSE)
    gx3 = cv2.Sobel(img_g3, cv2.CV_32F, 1, 0, ksize=3)
    gy3 = cv2.Sobel(img_g3, cv2.CV_32F, 0, 1, ksize=3)
    gmag3 = np.sqrt(gx3**2 + gy3**2).astype(np.float32)
    gx3_n = (gx3 / (np.abs(gx3).max() + 1e-8)).astype(np.float32)
    gy3_n = (gy3 / (np.abs(gy3).max() + 1e-8)).astype(np.float32)
    gmag3_n = (gmag3 / (gmag3.max() + 1e-8)).astype(np.float32)

    # Laplacian
    lap = cv2.Laplacian(img_8u, cv2.CV_32F)
    lap_n = ((lap - lap.min()) /
             (lap.max() - lap.min() + 1e-8)).astype(np.float32)

    #  Group E: patch GLCM
    glcm_maps = _compute_glcm_patch_features(img_8u, cfg.GLCM_PATCH_SIZE)
    # Shape: (H, W, 4)

    #  Group F: spatial features
    scan_u8 = scan_mask.astype(np.uint8)

    dist_b = distance_transform_edt(scan_u8).astype(np.float32)
    dist_b = (dist_b / (dist_b.max() + 1e-8)).astype(np.float32)

    ri, ci = np.where(scan_mask)
    cr = ri.mean() if len(ri) > 0 else H / 2
    cc = ci.mean() if len(ci) > 0 else W / 2
    yy, xx = np.mgrid[0:H, 0:W]
    dist_c = np.sqrt((yy - cr)**2 + (xx - cc)**2).astype(np.float32)
    dist_c = (dist_c / (dist_c.max() + 1e-8)).astype(np.float32)

    # Stack all features
    feat = np.column_stack([
        # A
        mean_s.ravel(), std_s.ravel(),
        mean_m.ravel(), std_m.ravel(),
        mean_l.ravel(), std_l.ravel(),
        mean_xl.ravel(), std_xl.ravel(),
        # B
        intensity_ratio.ravel(),
        # C
        f_entropy.ravel(),
        f_var.ravel(),
        # D
        gx1_n.ravel(), gy1_n.ravel(), gmag1_n.ravel(),
        gx3_n.ravel(), gy3_n.ravel(), gmag3_n.ravel(),
        lap_n.ravel(),
        # E
        glcm_maps[:, :, 0].ravel(),   # contrast
        glcm_maps[:, :, 1].ravel(),   # homogeneity
        glcm_maps[:, :, 2].ravel(),   # energy
        glcm_maps[:, :, 3].ravel(),   # correlation
        # F
        dist_b.ravel(),
        dist_c.ravel(),
    ]).astype(np.float32)

    assert feat.shape == (H * W, N_FEATURES), \
        f"Shape mismatch: {feat.shape} vs ({H*W}, {N_FEATURES})"
    return feat


print(f"\n[FEATURE EXTRACTION]")
print(f"  {N_FEATURES} features: {FEATURE_NAMES}")

# Sanity check
_proc  = preprocess(load_image_raw(train_pairs[0][0]),
                    load_mask_raw(train_pairs[0][1]))
_feats = extract_features(_proc['img_proc'], _proc['scan_mask'])
print(f"  Check OK — cropped shape {_proc['img_proc'].shape}, "
      f"features {_feats.shape}")
del _proc, _feats


# SECTION 6: DATASET CONSTRUCTION  (train_pairs only)


def build_dataset(pairs: list) -> tuple:
    """
    Per image (3000 total):
        1500 muscle pixels
        1050 boundary negatives: background, 5px <= dist <= 50px from mask
         450 random negatives: background, dist > 50px, inside scan
    """
    rng = np.random.default_rng(cfg.RANDOM_SEED)
    X_list = []
    y_list = []
    skipped = 0
    n_total = cfg.N_MUSCLE + cfg.N_BOUNDARY_NEG + cfg.N_RANDOM_NEG

    print(f"\n[DATASET CONSTRUCTION]")
    print(f"  {len(pairs)} images × {n_total} pixels = "
          f"~{len(pairs)*n_total:,} samples")
    t0 = time.time()

    for i, (img_path, mask_path) in enumerate(pairs):
        try:
            img_raw = load_image_raw(img_path)
            mask_raw = load_mask_raw(mask_path)
            proc = preprocess(img_raw, mask_raw)
        except Exception as e:
            print(f"  SKIP {img_path.name}: {e}")
            skipped += 1
            continue

        mask_crop = proc['mask_cropped']
        scan_flat = proc['scan_mask'].ravel()
        feats = extract_features(proc['img_proc'], proc['scan_mask'])
        labels = mask_crop.ravel().astype(int)

        # Distance transform for hard negative mining
        dist_to_mask = distance_transform_edt(~mask_crop.astype(bool))
        dist_flat    = dist_to_mask.ravel()

        # Index sets
        muscle_idx = np.where(
            (labels == 1) & scan_flat
        )[0]
        boundary_neg_idx = np.where(
            (labels == 0) & scan_flat &
            (dist_flat >= cfg.BOUNDARY_MIN) &
            (dist_flat <= cfg.BOUNDARY_MAX)
        )[0]
        random_neg_idx = np.where(
            (labels == 0) & scan_flat &
            (dist_flat > cfg.BOUNDARY_MAX)
        )[0]

        n_m  = min(cfg.N_MUSCLE, len(muscle_idx))
        n_bn = min(cfg.N_BOUNDARY_NEG, len(boundary_neg_idx))
        n_rn = min(cfg.N_RANDOM_NEG, len(random_neg_idx))

        if n_m == 0:
            print(f"  SKIP {img_path.name}: no muscle pixels in scan")
            skipped += 1
            continue

        chosen_m = rng.choice(muscle_idx,       n_m,  replace=False)
        chosen_bn = (rng.choice(boundary_neg_idx, n_bn, replace=False)
                     if n_bn > 0 else np.array([], dtype=int))
        chosen_rn = (rng.choice(random_neg_idx,   n_rn, replace=False)
                     if n_rn > 0 else np.array([], dtype=int))

        chosen = np.concatenate([chosen_m, chosen_bn, chosen_rn])
        X_list.append(feats[chosen])
        y_list.append(labels[chosen])

        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(pairs)}  ({time.time()-t0:.1f}s)")

    X = np.vstack(X_list).astype(np.float32)
    y = np.concatenate(y_list).astype(int)

    print(f"\n  Done in {time.time()-t0:.1f}s | skipped={skipped}")
    print(f"  Total: {X.shape[0]:,} | "
          f"muscle={np.sum(y==1):,} | bg={np.sum(y==0):,}")
    return X, y


X_all, y_all = build_dataset(train_pairs)

X_tr, X_vp, y_tr, y_vp = train_test_split(
    X_all, y_all, test_size=0.15,
    random_state=cfg.RANDOM_SEED, stratify=y_all
)
print(f"  Pixel split -> train: {len(X_tr):,} | val: {len(X_vp):,}")


# SECTION 7: MODEL TRAINING


print(f"\n[MODEL TRAINING]")
clf = RandomForestClassifier(
    n_estimators=cfg.RF_N_ESTIMATORS,
    max_depth=cfg.RF_MAX_DEPTH,
    min_samples_leaf=cfg.RF_MIN_SAMPLES_LEAF,
    class_weight='balanced',
    random_state=cfg.RANDOM_SEED,
    n_jobs=cfg.RF_N_JOBS,
    oob_score=True,
)
t0 = time.time()
clf.fit(X_tr, y_tr)
print(f"  Done in {time.time()-t0:.1f}s | OOB: {clf.oob_score_:.4f}")

y_pred_vp = clf.predict(X_vp)
print(f"\n  Pixel-level validation:")
print(classification_report(y_vp, y_pred_vp,
                             target_names=['background', 'muscle']))

joblib.dump(clf, cfg.MODEL_PATH)
print(f"  Model saved -> {cfg.MODEL_PATH}")

imp_df = pd.DataFrame({
    'feature':    FEATURE_NAMES,
    'importance': clf.feature_importances_,
}).sort_values('importance', ascending=False)
print(f"\n  Feature importances:")
print(imp_df.to_string(index=False))


# SECTION 8: POSTPROCESSING


def postprocess_mask(raw_mask: np.ndarray) -> np.ndarray:
    mask = raw_mask.astype(bool)
    mask = binary_closing(mask, disk(cfg.MORPH_CLOSE_RADIUS))
    mask = binary_fill_holes(mask)
    mask = remove_small_objects(mask, min_size=cfg.MIN_COMPONENT_SIZE)
    labeled = label(mask)
    regions = regionprops(labeled)
    if regions:
        largest = max(regions, key=lambda r: r.area)
        mask    = (labeled == largest.label)
    else:
        mask = np.zeros_like(raw_mask, dtype=bool)
    return mask.astype(np.uint8)


# SECTION 9: METRICS


def dice(pred, gt):
    p, g  = pred.astype(bool), gt.astype(bool)
    inter = np.logical_and(p, g).sum()
    denom = p.sum() + g.sum()
    return 1.0 if denom == 0 else 2.0 * inter / denom

def iou(pred, gt):
    p, g  = pred.astype(bool), gt.astype(bool)
    inter = np.logical_and(p, g).sum()
    union = np.logical_or(p, g).sum()
    return 1.0 if union == 0 else inter / union

def pxacc(pred, gt):
    return float(np.mean(pred.astype(int) == gt.astype(int)))


# SECTION 10: ACSA CALIBRATION


def build_acsa_calibration(pairs, metadata) -> tuple:
    meta_dict = dict(zip(metadata['stem'], metadata['Area_cm2']))
    cal = {}
    for img_path, mask_path in pairs:
        stem = img_path.stem
        area = meta_dict.get(stem)
        if area is None or (isinstance(area, float) and np.isnan(area)):
            continue
        try:
            mask_raw = load_mask_raw(mask_path)
            pix = int(mask_raw.astype(bool).sum())
            if pix > 0:
                cal[stem] = float(area) / float(pix)
        except Exception:
            continue
    values = list(cal.values())
    median_cal = float(np.median(values)) if values else 0.001
    print(f"  Calibration from {len(cal)} images")
    print(f"  Median: {median_cal:.6f} cm²/pixel  "
          f"range [{min(values):.6f}, {max(values):.6f}]")
    return cal, median_cal


def compute_acsa(mask_orig: np.ndarray, cm2_per_pixel: float) -> float:
    return float(mask_orig.astype(bool).sum()) * cm2_per_pixel


print(f"\n[ACSA CALIBRATION]")
cal_dict, median_cal = build_acsa_calibration(train_pairs, metadata)


# SECTION 11: FULL INFERENCE


def predict_image(img_raw, clf, cm2_per_pixel) -> dict:
    orig_shape = img_raw.shape
    proc = preprocess(img_raw)
    r0, r1, c0, c1 = proc['crop_bbox']
    H, W = proc['img_proc'].shape

    feats = extract_features(proc['img_proc'], proc['scan_mask'])
    pred_flat = clf.predict(feats)
    pred_flat[~proc['scan_mask'].ravel()] = 0

    mask_raw_crop = pred_flat.reshape(H, W).astype(np.uint8)
    mask_refined_crop = postprocess_mask(mask_raw_crop)

    mask_orig = np.zeros(orig_shape, dtype=np.uint8)
    mask_orig[r0:r1, c0:c1] = mask_refined_crop

    return {
        'img_proc': proc['img_proc'],
        'scan_mask': proc['scan_mask'],
        'crop_bbox': (r0, r1, c0, c1),
        'mask_raw_crop': mask_raw_crop,
        'mask_refined_crop': mask_refined_crop,
        'mask_orig': mask_orig,
        'acsa_cm2': compute_acsa(mask_orig, cm2_per_pixel),
    }


# SECTION 12: EVALUATION


def evaluate_split(pairs, clf, cal_dict, median_cal, split_name) -> pd.DataFrame:
    print(f"\n[EVALUATION — {split_name.upper()} ({len(pairs)} images)]")
    results = []
    t0 = time.time()

    for i, (img_path, mask_path) in enumerate(pairs):
        try:
            img_raw  = load_image_raw(img_path)
            mask_gt  = load_mask_raw(mask_path)
            cp       = cal_dict.get(img_path.stem, median_cal)
            out      = predict_image(img_raw, clf, cp)
            pred     = out['mask_orig']
            results.append({
                'image': img_path.name,
                'split': split_name,
                'dice': round(dice(pred, mask_gt), 4),
                'iou': round(iou(pred,  mask_gt), 4),
                'accuracy': round(pxacc(pred, mask_gt), 4),
                'acsa_pred': round(out['acsa_cm2'], 4),
                'cm2pp': round(cp, 6),
            })
        except Exception as e:
            print(f"  ERROR {img_path.name}: {e}")

        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(pairs)}  ({time.time()-t0:.1f}s)")

    df = pd.DataFrame(results)
    if not df.empty:
        print(f"\n  {split_name.upper()} ")
        print(f"  Dice     : {df['dice'].mean():.4f} +- {df['dice'].std():.4f}")
        print(f"  IoU      : {df['iou'].mean():.4f}  +- {df['iou'].std():.4f}")
        print(f"  Accuracy : {df['accuracy'].mean():.4f} +- {df['accuracy'].std():.4f}")
    return df


val_df  = evaluate_split(val_pairs,  clf, cal_dict, median_cal, 'val')
test_df = evaluate_split(test_pairs, clf, cal_dict, median_cal, 'test')

all_df = pd.concat([val_df, test_df], ignore_index=True)
all_df.to_csv(cfg.OUTPUT_DIR / "evaluation_v5.csv", index=False)
with open(cfg.RESULTS_PATH, 'w') as f:
    json.dump({
        'val':  {'n': len(val_df),
                 'dice': float(val_df['dice'].mean()),
                 'iou':  float(val_df['iou'].mean()),
                 'acc':  float(val_df['accuracy'].mean())},
        'test': {'n': len(test_df),
                 'dice': float(test_df['dice'].mean()),
                 'iou':  float(test_df['iou'].mean()),
                 'acc':  float(test_df['accuracy'].mean())},
    }, f, indent=2)


# SECTION 13: VISUALIZATION


def show_segmentation(img_path, mask_path, clf, cal_dict, median_cal,
                      title=""):
    """
    4-panel visualization:
        Panel 1 — Original ultrasound (full resolution)
        Panel 2 — Ground truth mask
        Panel 3 — Predicted mask
        Panel 4 — Overlay: green fill=prediction, yellow contour=GT boundary
    """
    img_raw = load_image_raw(img_path)
    mask_gt = load_mask_raw(mask_path)
    cp = cal_dict.get(img_path.stem, median_cal)
    out = predict_image(img_raw, clf, cp)
    pred = out['mask_orig']

    d = dice(pred, mask_gt)
    j = iou(pred,  mask_gt)
    a = out['acsa_cm2']

    # Overlay panel
    img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_GRAY2RGB)
    overlay = img_rgb.copy()
    overlay[pred == 1] = [0, 200, 80]
    panel4 = cv2.addWeighted(overlay, 0.40, img_rgb, 0.60, 0)
    gt_uint8 = (mask_gt * 255).astype(np.uint8)
    contours, _ = cv2.findContours(gt_uint8, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(panel4, contours, -1, (255, 220, 0), 2)

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    axes[0].imshow(img_raw, cmap='gray')
    axes[0].set_title("Original Ultrasound", fontsize=11)
    axes[1].imshow(mask_gt, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title("Ground Truth Mask", fontsize=11)
    axes[2].imshow(pred, cmap='gray', vmin=0, vmax=1)
    axes[2].set_title(f"Predicted Mask\nACSA = {a:.2f} cm²", fontsize=11)
    axes[3].imshow(panel4)
    axes[3].set_title(
        f"Overlay  (green=pred, yellow=GT)\n"
        f"Dice={d:.3f}  IoU={j:.3f}", fontsize=10)
    for ax in axes:
        ax.axis('off')
    fig.suptitle(f"{title}{img_path.name}", fontsize=12, fontweight='bold')
    plt.tight_layout()
    ipy_display(fig)
    plt.close(fig)


def show_feature_importance(clf):
    imps = clf.feature_importances_
    idx = np.argsort(imps)[::-1]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(range(len(imps)), imps[idx], color='steelblue', edgecolor='white')
    ax.set_xticks(range(len(imps)))
    ax.set_xticklabels([FEATURE_NAMES[i] for i in idx],
                       rotation=45, ha='right', fontsize=8)
    ax.set_ylabel("Importance")
    ax.set_title("Random Forest Feature Importances — v5")
    plt.tight_layout()
    ipy_display(fig)
    plt.close(fig)


def show_metrics(val_df, test_df):
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for ax, metric, color in zip(
            axes,
            ['dice', 'iou', 'accuracy'],
            ['#4C72B0', '#DD8452', '#55A868']):
        data = [val_df[metric].dropna(), test_df[metric].dropna()]
        bp   = ax.boxplot(data, patch_artist=True, labels=['Val', 'Test'])
        for patch in bp['boxes']:
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_ylim([0, 1.05])
        ax.set_title(metric.capitalize(), fontsize=12)
        for k, d in enumerate(data, 1):
            ax.text(k, max(0.02, d.mean() - 0.08),
                    f'{d.mean():.3f}', ha='center',
                    fontsize=9, color='darkred', fontweight='bold')
    fig.suptitle("Segmentation Metrics — Val vs Test (v5)",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    ipy_display(fig)
    plt.close(fig)


def show_acsa_scatter(val_df, test_df, metadata):
    meta_dict = dict(zip(metadata['stem'], metadata['Area_cm2']))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, df, title in zip(axes, [val_df, test_df],
                              ['Validation', 'Test']):
        gt_list, pred_list = [], []
        for _, row in df.iterrows():
            stem = Path(row['image']).stem
            gt   = meta_dict.get(stem)
            if gt is not None and not (isinstance(gt, float) and
                                       np.isnan(float(gt))):
                gt_list.append(float(gt))
                pred_list.append(float(row['acsa_pred']))
        if gt_list:
            ax.scatter(gt_list, pred_list, alpha=0.55, s=22)
            lim = max(max(gt_list), max(pred_list)) * 1.1
            ax.plot([0, lim], [0, lim], 'r--', lw=1.2, label='Perfect')
            r = float(np.corrcoef(gt_list, pred_list)[0, 1])
            ax.text(0.05, 0.92, f'r = {r:.3f}',
                    transform=ax.transAxes, fontsize=10)
            ax.set_xlabel("GT Area_cm² (metadata)")
            ax.set_ylabel("Predicted ACSA cm²")
            ax.legend(fontsize=9)
        ax.set_title(f"ACSA — {title}")
    fig.suptitle("Predicted vs Ground Truth ACSA — v5",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    ipy_display(fig)
    plt.close(fig)


print("\n[VISUALIZATION]")
show_feature_importance(clf)
show_metrics(val_df, test_df)
show_acsa_scatter(val_df, test_df, metadata)

print("\n  Val set examples (3 images):")
for i, (ip, mp) in enumerate(val_pairs[:3]):
    show_segmentation(ip, mp, clf, cal_dict, median_cal,
                      title=f"[VAL {i+1}] ")

print("\n  Test set examples (3 images):")
for i, (ip, mp) in enumerate(test_pairs[:3]):
    show_segmentation(ip, mp, clf, cal_dict, median_cal,
                      title=f"[TEST {i+1}] ")


# SECTION 14: PREDICTION API


def predict_new_image(image_input, clf, median_cal: float) -> dict:
    """
    Segment a new unseen ultrasound image and compute ACSA.

    Args:
        image_input : file path (str/Path) or numpy uint8 grayscale array
        clf         : trained RandomForestClassifier
        median_cal  : median cm²/pixel from training calibration

    Returns:
        dict: mask_orig, acsa_cm2, img_proc, crop_bbox
    """
    if isinstance(image_input, (str, Path)):
        img_raw  = load_image_raw(Path(image_input))
        img_name = Path(image_input).name
    elif isinstance(image_input, np.ndarray):
        img_raw  = image_input.copy()
        if img_raw.ndim == 3:
            img_raw = cv2.cvtColor(img_raw, cv2.COLOR_BGR2GRAY)
        img_name = "array_input"
    else:
        raise TypeError("image_input must be a path or numpy array")

    out = predict_image(img_raw, clf, median_cal)

    print(f"\n[PREDICTION — {img_name}]")
    print(f"  Crop bbox : {out['crop_bbox']}")
    print(f"  Muscle px : {out['mask_orig'].sum():,}")
    print(f"  ACSA      : {out['acsa_cm2']:.4f} cm²")

    img_rgb  = cv2.cvtColor(img_raw, cv2.COLOR_GRAY2RGB)
    overlay  = img_rgb.copy()
    overlay[out['mask_orig'] == 1] = [0, 200, 80]
    panel_ov = cv2.addWeighted(overlay, 0.40, img_rgb, 0.60, 0)

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    axes[0].imshow(img_raw, cmap='gray')
    axes[0].set_title("Original Ultrasound")
    axes[1].imshow(out['mask_orig'], cmap='gray', vmin=0, vmax=1)
    axes[1].set_title("Predicted Mask")
    axes[2].imshow(panel_ov)
    axes[2].set_title(f"Overlay\nACSA = {out['acsa_cm2']:.3f} cm²")
    for ax in axes:
        ax.axis('off')
    fig.suptitle(f"Prediction: {img_name}", fontsize=12)
    plt.tight_layout()
    ipy_display(fig)
    plt.close(fig)

    return {
        'mask_orig': out['mask_orig'],
        'acsa_cm2':  out['acsa_cm2'],
        'img_proc':  out['img_proc'],
        'crop_bbox': out['crop_bbox'],
    }


# SECTION 15: SUMMARY


print("\n" + "=" * 65)
print("  PIPELINE COMPLETE — v5")
print("=" * 65)
print(f"\n  Split  : {len(train_pairs)} train | "
      f"{len(val_pairs)} val | {len(test_pairs)} test")
print(f"\n  VAL  -> Dice {val_df['dice'].mean():.4f} | "
      f"IoU {val_df['iou'].mean():.4f} | "
      f"Acc {val_df['accuracy'].mean():.4f}")
print(f"  TEST -> Dice {test_df['dice'].mean():.4f} | "
      f"IoU {test_df['iou'].mean():.4f} | "
      f"Acc {test_df['accuracy'].mean():.4f}")
print(f"\n  ACSA median calibration : {median_cal:.6f} cm²/pixel")
print(f"\n  To predict on a new image:")
print(f"    result = predict_new_image('scan.tif', clf, median_cal)")
print(f"    print(result['acsa_cm2'], 'cm²')")
print("=" * 65)